# T1 - Deep Learning 

Feito por: Louise e Luca


Numpy faz as operacoes de matrizes para nos, então nos temos que criar uma classe base (pytorch chama de tensor) 

tensor --> matriz pessos + matriz de gradiente 

In [198]:
import numpy as np

Unidade basica - matriz que guarda as informcoes para a fazer o back propagation depois 

In [199]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.random.random(dimencao)

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        return self.peso + outra.peso

In [200]:
dimencao = [2,3]
exemplo = Tensor(dimencao)

print(exemplo)
print(exemplo.gradiente)

[[0.03753316 0.59585407 0.44488503]
 [0.0837026  0.2862656  0.2695741 ]]
[[0.76430757 0.4529702  0.62439765]
 [0.365876   0.26881445 0.31012854]]


In [201]:
exemplo2 = Tensor(dimencao)

In [202]:
exemplo + exemplo2

array([[0.58156741, 1.58781093, 0.85610064],
       [0.69475356, 1.11711196, 0.37530042]])

Agora vamos mexer no gradiente 
queremos impemetar as funcoes basicas para qaundo fizermos o back propagetion qualquer funcao ser diferenciavel (derivar )

uma forma de ccalular o gradiente: graudar as regras da derivada 


In [203]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.random.random(dimencao)
        self.regra = 0

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        self.regra = 1 
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso + outra.peso

        return retorno

    def __mul__(self, outra):  # ponto a ponto 

        self.regra = outra.peso
        outra.regra = self.peso


        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso * outra.peso
 
        return retorno
    

    

In [204]:
dimencao = [2,2]
teste1 = np.array([[1,2],[3,4]])
teste2 = np.array([[1,1],[1,1]])

exemplo_3 = Tensor(dimencao)
exemplo_4 = Tensor(dimencao)

exemplo_3.peso = teste1
exemplo_4.peso = teste2

print(exemplo_3, "\n\n", exemplo_4)

[[1 2]
 [3 4]] 

 [[1 1]
 [1 1]]


In [205]:
exemplo_4 * exemplo_3

In [206]:
exemplo_3.regra

array([[1, 1],
       [1, 1]])

In [207]:
exemplo_4.regra

array([[1, 2],
       [3, 4]])

Agora temos o nosso gradiente local, nos chamamos isso de regra, aqui esta uma teste para ver se esta funcionando:

dervada em relacao a 3 da funcao: ( (3 * 4) * 5 ) = 20 

In [208]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

print(exemplo_num3, "\n", exemplo_num4, "\n", exemplo_num5)
print( exemplo_num3 * exemplo_num4 * exemplo_num5)

print(exemplo_num5.regra) # 3 * 4 = 12 o gradiente esta certo
exemplo_num5.gradiente = exemplo_num5.regra 

print(exemplo_num5.gradiente)

print(exemplo_num3.regra)
exemplo_num3.gradiente = exemplo_num3.regra * exemplo_num5.peso

print(exemplo_num3.gradiente)

[3] 
 [4] 
 [5]
[60]
[12]
[12]
[4]
[20]


In [209]:
exemplo_3.peso.shape

(2, 2)

Agora nos queremos que atualize o gradiente automaticamente, sem ter que fazer 
```python 
exemplo_num3.gradiente = exemplo_num3.regra * exemplo_num5.peso
```

para isso a gente vai mudar para fazwe uma funcao= gradiente local * outro

In [210]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda x:0
        self.prox = None

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso + outra.peso
        self.prox = retorno # agora temos um grafo

        def regra(tensor):
            tensor.gradiente += self.prox.gradiente

        self.regra = lambda: regra(self)
        outra.regra = lambda: regra(outra)

        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso * outra.peso
        self.prox = retorno

        def regra():
            self.gradiente += self.prox.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += outra.prox.gradiente * self.peso 
 
        return retorno

    def backward(self): # calulo dos gradientes
        pass



    

In [211]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + exemplo_num4
teste.gradiente = np.ones_like(teste, float) 
exemplo_num3.regra()
print(exemplo_num3.gradiente)

[1.]


Vamos trabalhar numa forma coesa de fazer o backward pass automaticmaente, depois voltamos a implementar funções. Usaremos o algoritmo de ordenação topologica baseado em DFS. Para isso vamos mudar a lógica de prox, e definir um anterior, pois é consideravelmente mais facil trabalhar na ordem inversa

In [212]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        retorno = Tensor(self.peso.shape, (self, outra))
        retorno.peso = self.peso + outra.peso
        self.prox = retorno # agora temos um grafo

        def regra():
            self.gradiente += retorno.gradiente
            outra.gradiente += retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        retorno = Tensor(self.peso.shape, (self, outra))
        retorno.peso = self.peso * outra.peso
        self.prox = retorno

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno

    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [213]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + exemplo_num4
final = exemplo_num5 * teste
final.backward()
print(exemplo_num3.gradiente)

[5.]


Agora vamos completar as 4 operações basicas e permitir operações com escalares

In [214]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # AS duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = outra
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente
            outra.gradiente += retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = outra
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        return self.__mul__( other**-1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
        

    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [215]:
dimencao = [1]
num3 = np.array([3,4,5])
num4 = np.array([4,8,9])
num5 = np.array([5])

exemplo_num3 = Tensor([3])
exemplo_num4 = Tensor([3])
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 * 10
print(exemplo_num3)

[3. 4. 5.]


Agora temos as operações básicas, mas isso quebra o nosso sistema de backward pass, por causa de erros de dimensão, então precisamos adaptar as dimensõoes

In [216]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # As duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)
        
        
    # Função responsavel por permitir o backward pass mesmo sob broadcast
    def soma_dimensional(self, gradiente, shape):
        while gradiente.ndim > len(shape): # Isso elimina dimensões extras adicionadas via boradcasting pelo numpy
            gradiente = gradiente.sum(axis=0)
            
        for i, dim in enumerate(shape): # Isso reduz a dimensionalidade de dimensões existentes de volta para 1 (caso originalmente fosse um)
            if dim == 1 and gradiente.shape[i] != 1:
                gradiente = gradiente.sum(axis=i, keepdims=True) # keepdims = True mantem a dimensão original do tensor, na pratica ele adiciona um parenteses e mantem como dimensão de tamanho 1
                
        return gradiente
            
        

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente, self.peso.shape)
            outra.gradiente += self.soma_dimensional(retorno.gradiente, outra.peso.shape)
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        return self.__mul__( other**-1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
        

    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [217]:
a = np.array([[[1,2,3],[4,5,6]], [[7,8,9], [20,12,11]]])

a.sum(axis=1, keepdims=True).shape

(2, 1, 3)

In [218]:
dimencao = [1]
num3 = np.array([3,4,5])
num4 = np.array([4,8,9])
num5 = np.array([5])

exemplo_num3 = Tensor([3])
exemplo_num4 = Tensor([3])
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + 10
teste.backward()
print(exemplo_num5.gradiente)

[0.]


In [219]:
exemplo_num3 = Tensor([3])
exemplo_num3.peso = np.array([3,4,5])

a = exemplo_num3 ** (0.5)
print(a)

[1.73205081 2.         2.23606798]


Agora,temos que implementar as últimas funções que ainda faltam, que são produto matricial, log, radicialização e somatorio

In [220]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # As duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)
        
        
    # Função responsavel por permitir o backward pass mesmo sob broadcast
    def soma_dimensional(self, gradiente, shape):
        while gradiente.ndim > len(shape): # Isso elimina dimensões extras adicionadas via boradcasting pelo numpy
            gradiente = gradiente.sum(axis=0)
            
        for i, dim in enumerate(shape): # Isso reduz a dimensionalidade de dimensões existentes de volta para 1 (caso originalmente fosse um)
            if dim == 1 and gradiente.shape[i] != 1:
                gradiente = gradiente.sum(axis=i, keepdims=True) # keepdims = True mantem a dimensão original do tensor, na pratica ele adiciona um parenteses e mantem como dimensão de tamanho 1
                
        return gradiente
            
        

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente, self.peso.shape)
            outra.gradiente += self.soma_dimensional(retorno.gradiente, outra.peso.shape)
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente * outra.peso, self.peso.shape) # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += self.soma_dimensional(retorno.gradiente * self.peso, outra.peso.shape) 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        return self.__mul__( other**-1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
    
    def root(self, value = 2): # Podemos aproveitar a implementação em potencia para fazer a radicialização
        return self ** (1/value)
    
    def __matmul__(self, outra):
        a = self.peso @ outra.peso
        retorno = Tensor(a.shape, (self, outra))
        retorno.peso = a
        
        def regra():
            self.gradiente +=retorno.gradiente @ outra.peso.T
            outra.gradiente += self.peso.T @ retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    
    def log(self, base="euler"):
        if base == "euler":
            resultado = np.log(self.peso)
            retorno = Tensor(resultado.shape, (self,))
            retorno.peso = resultado
            
            def regra():
                self.gradiente += retorno.gradiente/self.peso
                
            retorno.regra = regra
            
            return retorno
        
        resultado = np.log(self.peso) / np.log(base) # Propriedade de mudança de base dos logaritmos
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente / (self.peso * np.log(base))

        retorno.regra = regra

        return retorno
    
    def sum(self, axis=0):
        resultado = np.sum(self.peso, axis=axis)
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            
            gradiente = retorno.gradiente
            
            if axis is not None: # Isso é para impedir o bug de caso o resultado final do sum no foarward tenha sido um escalar
                # Expandindo a dimensão removida durante o foarward
                gradiente = np.expand_dims(gradiente, axis=axis)
            
            self.gradiente += np.broadcast_to(gradiente, self.peso.shape) # Copia as dimensões em 1 para bater com o valor da matriz
            
            
        retorno.regra = regra
        
        return retorno
        

        
        
            
    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [221]:
exemplo_num3 = Tensor([3,2])
exemplo_num4 = Tensor([2,3])

exemplo_num3.peso = np.array([[1,2],
                              [3,4], 
                              [5,6]])
exemplo_num4.peso = np.array([[1,2,3], [4,5,6]])

teste = exemplo_num3 @ exemplo_num4
teste2 = teste + teste
print(teste)
print(teste2)



[[ 9. 12. 15.]
 [19. 26. 33.]
 [29. 40. 51.]]
[[ 18.  24.  30.]
 [ 38.  52.  66.]
 [ 58.  80. 102.]]


In [222]:
exemplo_num3 = Tensor([3,2])
exemplo_num4 = Tensor([2,3])

exemplo_num3.peso = np.array([[1,2],
                              [3,4], 
                              [5,6]])
exemplo_num4.peso = np.array([[1,2,3], [4,5,6]])

teste = exemplo_num3.log(2).sum()
teste.backward()
print(teste)
print(teste.gradiente)



[3.9068906 5.5849625]
[1. 1.]


Abaixo seguem apenas testes e detlahes da adaptação do autograd para ele ter as ferramentas para facilitar a criação de mlps

In [223]:
import numpy as np

class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.zeros(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # As duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)
        
        
    # Função responsavel por permitir o backward pass mesmo sob broadcast
    @staticmethod
    def soma_dimensional(gradiente, shape):
        while gradiente.ndim > len(shape): # Isso elimina dimensões extras adicionadas via boradcasting pelo numpy
            gradiente = gradiente.sum(axis=0)
            
        for i, dim in enumerate(shape): # Isso reduz a dimensionalidade de dimensões existentes de volta para 1 (caso originalmente fosse um)
            if dim == 1 and gradiente.shape[i] != 1:
                gradiente = gradiente.sum(axis=i, keepdims=True) # keepdims = True mantem a dimensão original do tensor, na pratica ele adiciona um parenteses e mantem como dimensão de tamanho 1
                
        return gradiente
            
        

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente, self.peso.shape)
            outra.gradiente += self.soma_dimensional(retorno.gradiente, outra.peso.shape)
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente * outra.peso, self.peso.shape) # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += self.soma_dimensional(retorno.gradiente * self.peso, outra.peso.shape) 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        if isinstance(other, (int, float, np.number)):
            return self * (1 / other)
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
    
    def __matmul__(self, outra):
        a = self.peso @ outra.peso
        retorno = Tensor(a.shape, (self, outra))
        retorno.peso = a
        
        def regra():
            self.gradiente +=retorno.gradiente @ outra.peso.T
            outra.gradiente += self.peso.T @ retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    
    def log(self, base="euler"):
        if base == "euler":
            resultado = np.log(self.peso)
            retorno = Tensor(resultado.shape, (self,))
            retorno.peso = resultado
            
            def regra():
                self.gradiente += retorno.gradiente/self.peso
                
            retorno.regra = regra
            
            return retorno
        
        resultado = np.log(self.peso) / np.log(base) # Propriedade de mudança de base dos logaritmos
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente / (self.peso * np.log(base))

        retorno.regra = regra

        return retorno

    def root(self, value = 2): # Podemos aproveitar a implementação em potencia para fazer a radicialização
            return self ** (1/value)
    
    def sum(self, axis=None):
        resultado = np.sum(self.peso, axis=axis)
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            
            gradiente = retorno.gradiente
            
            if axis is not None: # Isso é para impedir o bug de caso o resultado final do sum no foarward tenha sido um escalar
                # Expandindo a dimensão removida durante o foarward
                gradiente = np.expand_dims(gradiente, axis=axis)
            
            self.gradiente += np.broadcast_to(gradiente, self.peso.shape) # Copia as dimensões em 1 para bater com o valor da matriz
            
            
        retorno.regra = regra
        
        return retorno
    
    
    
    # Funções de ativação

    def reLU(self):
        retorno = Tensor(self.peso.shape, (self,))
        retorno.peso = np.where(self.peso>0, self.peso, 0)

        def regra():
            self.gradiente += (self.peso > 0) * retorno.gradiente

        retorno.regra = regra

        return retorno
    
    
    def sigmoid(self):
        resultado = 1 / (1 + np.exp(-self.peso))

        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente * retorno.peso * (1 - retorno.peso)

        retorno.regra = regra

        return retorno
    
    
    def tanh(self):
        resultado = np.tanh(self.peso)
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            self.gradiente += (1 - resultado**2) * retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    
    def linear(self): # Essa aqui foi difícil 
        return self
    
    def leakyReLU(self, alpha=0.01):
        resultado = np.where(self.peso>0, self.peso, alpha*self.peso)
        retorno = Tensor(self.peso.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            self.gradiente += np.where(self.peso>0, 1, alpha) * retorno.gradiente
            
        retorno.regra = regra
        
        return retorno

            
    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()
            
    def __call__(self, outro):
        return self @ outro
    
    def __getitem__(self, indice): # Permite indexarmos um vetor como no numpy, ou sej podemos ter tensor[2:, 1]
        resultado = self.peso[indice]

        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        return retorno


def Dense(input, output, inicializacao = "He", limite_inferior=0, limite_superior=1, desvio_padrao=1, media=0):
    matriz = Tensor([input, output])
    
    if inicializacao == "He":
        matriz.peso = np.random.normal(0,
                                       np.sqrt( 2/input),
                                       size=(input,output))
    elif inicializacao == "Xavier":
        matriz.peso = np.random.normal(0,
                                       np.sqrt( 2/(input+output)),
                                       size=(input,output))
                                       
    elif inicializacao == "Uniforme":
        matriz.peso = np.random.uniform(limite_inferior, limite_superior, size=(input, output))
    elif inicializacao == "Normal":
        matriz.peso = np.random.normal(media,
                                       np.sqrt(desvio_padrao), 
                                       size=(input, output))
        
    return matriz


class MLP:
    def __init__(self, *camadas):
        self.camadas = camadas
        self.bias = [Tensor((1,pesos.peso.shape[1])) for pesos,_,_ in camadas]

    def __call__(self, x):
        for i, (pesos, tem_bias, ativacao) in enumerate(self.camadas):
            if tem_bias:
                x = x @ pesos + self.bias[i]
            else:
                x = x @ pesos
            x = getattr(x, ativacao)()

        return x
    
    def parametros(self):
        retorno = [pesos for pesos, _, _ in self.camadas] + self.bias # Quem diria que somar listas na verdade concatena elas, boa python
        return retorno
    
    
    def zerar_gradiente(self):
        for pesos in self.parametros():
            pesos.gradiente = np.zeros_like(pesos.gradiente)
            
            
class SGD:
    def __init__(self, mlp, lr):
        self.lr = lr
        self.mlp = mlp
        
    def passo(self):
        for parametro in self.mlp.parametros():
            parametro.peso = parametro.peso - self.lr * parametro.gradiente
            
class Adam:
    def __init__(self, mlp, lr, beta1 = 0.9, beta2 = 0.999, e = 0.0000001):
        self.lr = lr
        self.mlp = mlp
        self.beta1 = beta1
        self.beta2 = beta2
        self.e = e
        self.m = [
            np.zeros_like(p.peso) for p in self.mlp.parametros()
        ]

        self.v = [
            np.zeros_like(p.peso) for p in self.mlp.parametros()
        ]
        self.estado = 0
        
    def passo(self):
        self.estado += 1 # iniciando em cima, não preciso somar no expoente durante cada for
        
        for i,parametro in enumerate(self.mlp.parametros()):
            self.m[i] = self.beta1 * self.m[i] + (1-self.beta1) * (parametro.gradiente)
            self.v[i] = self.beta2 * self.v[i] + (1-self.beta2) * ((parametro.gradiente)**2)
            
            m_adam = self.m[i]/(1 - self.beta1**self.estado)
            v_adam = self.v[i]/(1 - self.beta2**self.estado)
            
            parametro.peso = parametro.peso - self.lr * m_adam / (np.sqrt(v_adam) + self.e)
            
        
    
        
def MSE(y, y_pred):
    custo = ((y - y_pred)**2).sum()
    return custo / y.peso.size

def BCE(y, y_pred):
    custo = -(y * y_pred.log() + (1-y) * (1-y_pred).log())
    return custo.sum() / y.peso.size



In [224]:
exemplo_num3 = Tensor([3,2])
exemplo_num4 = Tensor([2,3])

exemplo_num3.peso = np.array([[1,2],
                              [3,4], 
                              [5,6]])
exemplo_num4.peso = np.array([[1,2,3], [4,5,6]])

teste = exemplo_num3.sigmoid()
teste.backward()
print(teste)
print(teste.gradiente)



[[0.73105858 0.88079708]
 [0.95257413 0.98201379]
 [0.99330715 0.99752738]]
[[1. 1.]
 [1. 1.]
 [1. 1.]]


In [ ]:
modelo = MLP(
    (Dense(3, 32, "He"), True, "reLU"),
    (Dense(32, 32, "He"), True,"reLU"),
    (Dense(32, 32, "He"), True,"reLU"),
    (Dense(32, 4, "He"), True, "reLU"),
    (Dense(4, 1, "He"), True,"linear")
)

otimizador = Adam(modelo, 0.001)

tamanho = (10000, 3)

x = Tensor(tamanho)
x.peso = np.random.randint(-30, 100, size=tamanho)

y = Tensor((tamanho[0], 1))
y.peso = (
    x.peso[:, 0:1] * 10
    - x.peso[:, 1:2] * 2
    + x.peso[:, 2:3]
)

for epoch in range(100):
    todas_loss = []

    for linha in range(tamanho[0]):
        entrada = Tensor((1, 3))
        entrada.peso = x.peso[linha:linha+1, :]

        y_real = Tensor((1, 1))
        y_real.peso = y.peso[linha:linha+1, :]

        modelo.zerar_gradiente()

        y_pred = modelo(entrada)

        loss = MSE(y_real, y_pred)

        loss.backward()
        otimizador.passo()

        todas_loss.append(float(loss.peso))

    print("Epoca:", epoch, "- Loss:", sum(todas_loss) / len(todas_loss))

/tmp/ipykernel_1104841/2680260315.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  todas_loss.append(float(loss.peso))


Epoca: 0 - Loss: 2403.4668877302347
Epoca: 1 - Loss: 79.38708570105454
Epoca: 2 - Loss: 47.79828677357678
Epoca: 3 - Loss: 30.617011814497918
Epoca: 4 - Loss: 26.471752777865134
Epoca: 5 - Loss: 23.29291842968419
Epoca: 6 - Loss: 24.772172431919234
Epoca: 7 - Loss: 18.335672554090284
Epoca: 8 - Loss: 14.19048134605434
Epoca: 9 - Loss: 17.654442030309887
Epoca: 10 - Loss: 10.910436945068964


In [ ]:
modelo = MLP(
    (Dense(3, 1, "Xavier"), "linear")
)

otimizador = Adam(modelo, 0.01)

for epoch in range(3000):
    modelo.zerar_gradiente()

    y_pred = modelo(x)
    loss = MSE(y, y_pred)

    loss.backward()
    otimizador.passo()

    if epoch % 100 == 0:
        print(epoch, loss.peso)

0 [224742.0034383]
100 [156555.65661875]
200 [117256.05307584]
300 [90864.06582397]
400 [70645.5640043]
500 [54794.70553858]
600 [42328.49516315]
700 [32466.70416357]
800 [24632.0644324]
900 [18418.99873073]
1000 [13535.27249387]
1100 [9752.66867655]
1200 [6877.94087848]
1300 [4740.33321105]
1400 [3188.31187351]
1500 [2089.83279337]
1600 [1333.06435583]
1700 [826.33104221]
1800 [497.02197196]
1900 [289.63336386]
2000 [163.26019461]
2100 [88.86856221]
2200 [46.63316278]
2300 [23.54633893]
2400 [11.41803929]
2500 [5.30648296]
2600 [2.35841986]
2700 [1.00006008]
2800 [0.40359677]
2900 [0.15461131]
